# 04 — Phase 5: Multimodal Feature Fusion Network Construction & Training
### Multimodal Deepfake Forensic Workbench

**Goal of this notebook:**
1. Load pre-extracted feature payloads from `features/`:
   - $F_{image} \in \mathbb{R}^{1792}$ (EfficientNet-B4)
   - $F_{video} \in \mathbb{R}^{256}$ (ResNet-18 + DirectMLLSTM)
   - $F_{audio} \in \mathbb{R}^{768}$ (WavLM)
2. Construct balanced multimodal feature triplets $(F_{image}, F_{video}, F_{audio}, \text{label})$ for Real and Fake classes.
3. Build **MultimodalFusionNetwork** with projection layers, adaptive attention weighting, and a deep classification head.
4. Train for 15 epochs using AdamW optimizer and evaluate Accuracy, Precision, Recall, F1-score, and ROC-AUC.
5. Save trained model checkpoint to `models/multimodal_fusion.pt`.

In [ ]:
import os, json, random, time, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Cap CPU threads to keep CPU cool (<60°C)
torch.set_num_threads(2)

CONFIG_PATH = "shared_config.json"
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r") as f:
        CFG = json.load(f)
else:
    CFG = {"project_root": ".", "seed": 42}

PROJECT_ROOT = CFG.get("project_root", ".")
SEED = CFG.get("seed", 42)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    import torch_directml
    device = torch_directml.device()
    print("Device: DirectML GPU (AMD Radeon RX 580)", device)
except ImportError:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)


## 1. Load Pre-Extracted Feature Payloads & Build Multimodal Triplets

In [ ]:
features_dir = os.path.join(PROJECT_ROOT, "features")
img_path = os.path.join(features_dir, "image_features.pt")
vid_path = os.path.join(features_dir, "video_features.pt")
aud_path = os.path.join(features_dir, "audio_features.pt")

print("Loading feature payloads from features/...")
img_data = torch.load(img_path, map_location="cpu", weights_only=False)
vid_data = torch.load(vid_path, map_location="cpu", weights_only=False)
aud_data = torch.load(aud_path, map_location="cpu", weights_only=False)

def split_by_label(feat_dict):
    reals, fakes = [], []
    for k, v in feat_dict.items():
        feat = v["feature"].float()
        lbl = int(v["label"])
        if lbl == 0:
            reals.append(feat)
        else:
            fakes.append(feat)
    return reals, fakes

img_reals, img_fakes = split_by_label(img_data)
vid_reals, vid_fakes = split_by_label(vid_data)
aud_reals, aud_fakes = split_by_label(aud_data)

print(f"Image Modality  -> Real: {len(img_reals):,}, Fake: {len(img_fakes):,}")
print(f"Video Modality  -> Real: {len(vid_reals):,}, Fake: {len(vid_fakes):,}")
print(f"Audio Modality  -> Real: {len(aud_reals):,}, Fake: {len(aud_fakes):,}")

class MultimodalFusionDataset(Dataset):
    def __init__(self, img_r, img_f, vid_r, vid_f, aud_r, aud_f, n_per_class=5000, mask_prob=0.15):
        self.samples = []
        self.mask_prob = mask_prob
        
        # Real samples
        for i in range(n_per_class):
            fi = img_r[i % len(img_r)]
            fv = vid_r[i % len(vid_r)]
            fa = aud_r[i % len(aud_r)]
            self.samples.append((fi, fv, fa, 0))
            
        # Fake samples
        for i in range(n_per_class):
            fi = img_f[i % len(img_f)]
            fv = vid_f[i % len(vid_f)]
            fa = aud_f[i % len(aud_f)]
            self.samples.append((fi, fv, fa, 1))
            
        random.shuffle(self.samples)
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        fi, fv, fa, label = self.samples[idx]
        
        # Modality Dropout / Masking during training for robustness
        if self.mask_prob > 0 and random.random() < self.mask_prob:
            m_idx = random.randint(0, 2)
            if m_idx == 0:
                fi = torch.zeros_like(fi)
            elif m_idx == 1:
                fv = torch.zeros_like(fv)
            else:
                fa = torch.zeros_like(fa)
                
        return fi, fv, fa, torch.tensor(label, dtype=torch.long)

full_fusion_dataset = MultimodalFusionDataset(img_reals, img_fakes, vid_reals, vid_fakes, aud_reals, aud_fakes, n_per_class=5000)
val_size = int(0.2 * len(full_fusion_dataset))
train_size = len(full_fusion_dataset) - val_size
train_ds, val_ds = random_split(full_fusion_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

print(f"\nMultimodal Fusion Dataset Ready: {len(train_ds):,} Train samples | {len(val_ds):,} Val samples")


## 2. Multimodal Fusion Network Architecture

- **Modality Projection Layers**: Maps heterogeneous feature dimensions ($1792, 256, 768$) into a normalized $256$-dimensional embedding space.
- **Adaptive Attention Weighting**: Dynamically computes modality importance weights $(\alpha_{img}, \alpha_{vid}, \alpha_{aud})$.
- **Deep Classification Head**: Concatenates weighted projections into a $768$-dim joint embedding and classifies Real vs Fake.

In [ ]:
class ModalityProjection(nn.Module):
    def __init__(self, in_dim, proj_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
    def forward(self, x):
        return self.net(x)

class MultimodalFusionNetwork(nn.Module):
    def __init__(self, img_dim=1792, vid_dim=256, aud_dim=768, proj_dim=256, num_classes=2):
        super().__init__()
        self.proj_img = ModalityProjection(img_dim, proj_dim)
        self.proj_vid = ModalityProjection(vid_dim, proj_dim)
        self.proj_aud = ModalityProjection(aud_dim, proj_dim)
        
        # Adaptive Modality Attention Gate
        self.attn_gate = nn.Sequential(
            nn.Linear(proj_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
            nn.Softmax(dim=-1)
        )
        
        # Deep Fusion Classifier Head
        fused_dim = proj_dim * 3
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
        
    def extract_fusion_embedding(self, f_img, f_vid, f_aud):
        p_img = self.proj_img(f_img)
        p_vid = self.proj_vid(f_vid)
        p_aud = self.proj_aud(f_aud)
        
        concat_feat = torch.cat([p_img, p_vid, p_aud], dim=-1)
        attn_weights = self.attn_gate(concat_feat)  # (B, 3)
        
        # Weighted feature representations
        p_img_w = p_img * attn_weights[:, 0:1]
        p_vid_w = p_vid * attn_weights[:, 1:2]
        p_aud_w = p_aud * attn_weights[:, 2:3]
        
        fused = torch.cat([p_img_w, p_vid_w, p_aud_w], dim=-1)
        return fused, attn_weights
        
    def forward(self, f_img, f_vid, f_aud):
        fused, _ = self.extract_fusion_embedding(f_img, f_vid, f_aud)
        return self.classifier(fused)

model = MultimodalFusionNetwork().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MultimodalFusionNetwork Initialized!")
print(f"Total Trainable Parameters: {total_params:,}")


## 3. Train Multimodal Fusion Network

In [ ]:
EPOCHS = 15
LR = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Starting Multimodal Fusion Training for {EPOCHS} Epochs on {device}...\n", flush=True)
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_n = 0.0, 0, 0
    
    for fi, fv, fa, labels in train_loader:
        fi, fv, fa, labels = fi.to(device), fv.to(device), fa.to(device), labels.to(device)
        
        outputs = model(fi, fv, fa)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        bs = labels.size(0)
        train_loss += loss.item() * bs
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_n += bs
        
    scheduler.step()
    
    # Validation
    model.eval()
    val_loss, val_correct, val_n = 0.0, 0, 0
    with torch.no_grad():
        for fi, fv, fa, labels in val_loader:
            fi, fv, fa, labels = fi.to(device), fv.to(device), fa.to(device), labels.to(device)
            outputs = model(fi, fv, fa)
            loss = criterion(outputs, labels)
            bs = labels.size(0)
            val_loss += loss.item() * bs
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_n += bs
            
    tr_acc = train_correct / train_n
    va_acc = val_correct / val_n
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss/train_n:.4f} | Train Acc: {tr_acc:.4f} | Val Loss: {val_loss/val_n:.4f} | Val Acc: {va_acc:.4f}", flush=True)

elapsed = time.time() - t0
print(f"\nTraining Complete in {elapsed:.2f} seconds!")


## 4. Full Evaluation & Performance Metrics

In [ ]:
model.eval()
all_preds, all_probs, all_targets = [], [], []

with torch.no_grad():
    for fi, fv, fa, labels in val_loader:
        fi, fv, fa = fi.to(device), fv.to(device), fa.to(device)
        outputs = model(fi, fv, fa)
        probs = F.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_targets.extend(labels.numpy())

acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds)
rec = recall_score(all_targets, all_preds)
f1 = f1_score(all_targets, all_preds)
roc_auc = roc_auc_score(all_targets, all_probs)
cm = confusion_matrix(all_targets, all_preds)

print("==========================================================")
print("    MULTIMODAL FUSION PERFORMANCE METRICS METRICS REPORT   ")
print("==========================================================")
print(f" Accuracy   : {acc * 100:.2f}%")
print(f" Precision  : {prec * 100:.2f}%")
print(f" Recall     : {rec * 100:.2f}%")
print(f" F1-Score   : {f1 * 100:.2f}%")
print(f" ROC-AUC    : {roc_auc:.4f}")
print("----------------------------------------------------------")
print("Confusion Matrix:")
print(f"  [[TN: {cm[0][0]:<5} FP: {cm[0][1]:<5}]")
print(f"   [FN: {cm[1][0]:<5} TP: {cm[1][1]:<5}]]")
print("==========================================================")


## 5. Save Multimodal Fusion Model Checkpoint

In [ ]:
models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, "multimodal_fusion.pt")

payload = {
    "model_state_dict": model.state_dict(),
    "proj_dim": 256,
    "img_dim": 1792,
    "vid_dim": 256,
    "aud_dim": 768,
    "metrics": {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": roc_auc
    }
}

torch.save(payload, save_path)
size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"SUCCESS: Saved Multimodal Fusion Checkpoint at: {save_path} ({size_mb:.2f} MB)")
